# Lab 01 — AI Model Detective

Runs on CPU. Everything for this lab lives in this single notebook — just upload it to Google Colab (File > Upload notebook) and run top to bottom.

## The problem

> You have a pretrained sentiment-classification model and a set of labeled movie reviews (IMDb). Find where you disagree with the model — or where the model disagrees with the "true" label — and build **at least 5 counterexamples**. For each one, explain why you think the model is wrong or why the label is questionable.

You are not required to train anything this week. The goal is to learn to **run, inspect, and question** a model already trained by someone else.

## Acceptance criteria

1. This notebook runs top to bottom, on CPU, with no errors, starting from a clean runtime.
2. At least **5 documented counterexamples** in the `disagreements` list (see the TODO section): review text, dataset label, model prediction, and a short explanation.
3. The acceptance-check cell near the end passes.
4. The notebook ends with the `RESULTS` block filled in.

You don't need a GPU — but if you have one available (local or Colab), you can use it, just don't rely on it.

## 1. Setup

In [ ]:
%pip install -q transformers datasets pandas scikit-learn

## 2. Load the data

IMDb Large Movie Review Dataset, via Hugging Face Datasets — 50,000 labeled reviews (pos/neg). We only need a small sample for this lab.

In [ ]:
import pandas as pd
from datasets import load_dataset

imdb = load_dataset("stanfordnlp/imdb", split="test")
df = imdb.shuffle(seed=42).select(range(300)).to_pandas()
df["label"] = df["label"].map({0: "neg", 1: "pos"})

print(df.shape)
df.head()

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/unsupervised-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 42.0MB            

... (2 outputs left out)

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

(300, 2)


,text,label
0,<br /><br />When I unsuspectedly rented A Thou...,pos
1,This is the latest entry in the long series of...,pos
2,This movie was so frustrating. Everything seem...,neg
3,"I was truly and wonderfully surprised at ""O' B...",pos
4,This movie spends most of its time preaching t...,neg


## 3. Load a pretrained model

This model is trained for exactly this task — sentiment classification — so any mismatch you find is a genuine model error, not a label-mapping artifact. See R2 in the README for a reference. This is just an example call; swap the model if you want.

In [ ]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=-1,  # forces CPU
)

sample = df.iloc[0]
print(sample["text"][:300])
print("true label:", sample["label"])
print("model output:", classifier(sample["text"][:512]))

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

<br /><br />When I unsuspectedly rented A Thousand Acres, I thought I was in for an entertaining King Lear story and of course Michelle Pfeiffer was in it, so what could go wrong?<br /><br />Very quickly, however, I realized that this story was about A Thousand Other Things besides just Acres. I sta
true label: pos
model output: [{'label': 'POSITIVE', 'score': 0.9975194334983826}]


## 4. TODO — find at least 5 disagreements

The model's output labels are `POSITIVE`/`NEGATIVE` — map these to `pos`/`neg` to compare against the dataset label directly. Your job:

1. Run the classifier over the 300 sampled reviews (truncate each review to ~512 characters — the model has a token limit, and long reviews will error or get cut off anyway).
2. Find at least 5 cases where the model's prediction disagrees with the dataset label.
3. Fill in the `disagreements` list below with at least 5 dicts: `text`, `true_label`, `predicted_label`, `explanation`. Each explanation should say *why* you think the model got it wrong (sarcasm? mixed review? mislabeled?) — not just that it disagreed.

In [ ]:
import time
import pandas as pd

texts = df["text"].fillna("").astype(str).tolist()
true_labels = df["label"].tolist()

model_inputs = [text[:512] for text in texts]

start_time = time.perf_counter()
outputs = [
    classifier(text, truncation=True)[0]
    for text in model_inputs
]
inference_seconds = time.perf_counter() - start_time

label_map = {"POSITIVE": "pos", "NEGATIVE": "neg"}
predicted_labels = [label_map[out["label"]] for out in outputs]

results_df = pd.DataFrame({
    "text": texts,
    "model_input": model_inputs,
    "true_label": true_labels,
    "predicted_label": predicted_labels,
    "confidence": [float(out["score"]) for out in outputs],
})
results_df["disagreement"] = (
    results_df["true_label"] != results_df["predicted_label"]
)
mismatches_df = results_df.loc[results_df["disagreement"]].copy()
mismatches_df = mismatches_df.reset_index(drop=True)

# masuratori pentru rezultate
reviews_analyzed = len(results_df)
disagreement_count = len(mismatches_df)
disagreement_rate = (
    float(results_df["disagreement"].mean()) if reviews_analyzed else 0.0
)
average_inference_seconds = (
    inference_seconds / reviews_analyzed if reviews_analyzed else 0.0
)
n_params = sum(p.numel() for p in classifier.model.parameters())
estimated_fp32_mib = n_params * 4 / (1024 ** 2)

print(f"Recenzii analizate: {reviews_analyzed}")
print(f"Dezacorduri găsite: {disagreement_count}")
print(f"Rata dezacordurilor: {disagreement_rate:.2%}")
print(f"Timp inferență: {inference_seconds:.2f} secunde")
print(f"Timp mediu/recenzie: {average_inference_seconds:.4f} secunde")
print(f"Parametri model: {n_params:,}")
print(f"Dimensiune estimată ponderi FP32: {estimated_fp32_mib:.2f} MiB")
print("\nPrimele 15 dezacorduri. Citește textul original înainte să alegi explicația.")

for idx, row in mismatches_df.head(15).iterrows():
    print("\n" + "=" * 80)
    print(f"ID dezacord: {idx}")
    print(f"IMDb: {row['true_label']} | Model: {row['predicted_label']}")
    print(f"Scorul clasei prezise: {row['confidence']:.4f}")
    print(f"Text original:\n{row['text']}")
    print(f"\nText trimis modelului (max. 512 caractere):\n{row['model_input']}")

# Ipoteze de verificat
def draft_explanation(row):
    text = row["text"]
    fragment = row["model_input"]
    lower = fragment.lower()

    sarcasm_markers = [
        "yeah right", "as if", "what a masterpiece", "worst", "so-called",
        "i'm sure", "of course", "brilliant", "wonderful"
    ]
    positive_words = [
        "great", "excellent", "love", "wonderful", "amazing", "good", "best"
    ]
    negative_words = [
        "bad", "awful", "boring", "waste", "terrible", "poor", "worst", "hate"
    ]

    if len(text) > len(fragment):
        return (
            "Ipoteza de verificat: recenzia a fost trunchiată la 512 caractere, "
            "deci o concluzie sau un contrast apărut mai târziu poate lipsi "
            "din contextul modelului."
        )
    if any(marker in lower for marker in sarcasm_markers):
        return (
            "Ipoteza de verificat: formularea poate fi ironică sau sarcastică; "
            "modelul ar putea interpreta literal cuvintele evaluative. "
            "Confirmă această interpretare citind contextul."
        )
    has_pos = any(word in lower for word in positive_words)
    has_neg = any(word in lower for word in negative_words)
    if has_pos and has_neg:
        return (
            "Ipoteza de verificat: fragmentul contine indicii atat pozitive, "
            "cat si negative, deci sentimentul poate fi mixt. Verifica ce "
            "opinie predomină în recenzie."
        )
    return (
        "Ipoteza de verificat: fragmentul poate sa nu exprime sentimentul "
        "in mod direct sau poate necesita context. Citeste recenzia și "
        "notează indiciul concret care sustine eticheta IMDb ori predictia."
    )

if len(mismatches_df) >= 5:
    disagreements = []
    for idx, row in mismatches_df.head(5).iterrows():
        disagreements.append({
            "text": row["text"],
            "true_label": row["true_label"],
            "predicted_label": row["predicted_label"],
            "explanation": draft_explanation(row),
        })
else:
    disagreements = []
    print(
        "\nAu fost găsite mai puțin de 5 dezacorduri. "
        "Acceptance check nu poate trece cu acest eșantion; inspectează "
        "mismatches_df și extinde eșantionul dacă instrucțiunile cursului permit."
    )

print(f"\nExemple pregătite pentru acceptance check: {len(disagreements)}")
print("IMPORTANT: editează explicațiile ca să reflecte ceea ce observi efectiv în fiecare recenzie.")

Recenzii analizate: 300
Dezacorduri găsite: 59
Rata dezacordurilor: 19.67%
Timp inferență: 66.10 secunde
Timp mediu/recenzie: 0.2203 secunde
Parametri model: 66,955,010
Dimensiune estimată ponderi FP32: 255.41 MiB

Primele 15 dezacorduri. Citește textul original înainte să alegi explicația.

ID dezacord: 0
IMDb: neg | Model: pos
Scorul clasei prezise: 0.9028
Text original:
The story is: a turn-of-the-century troupe of actors, along with producers and theatre-owners, have very complicated relationships. A resident playwright has written a psychological drama. He wants to get a good production on stage, but can't unless he convinces a pariticular reviewer to revisit the production, and give a positive review. If his production does not go on, then the troupe will put on The Doll House, recently written by Ibsen. Many different relationships among the principals are explored; none of them interesting. But the involvement of the characters with one another lead to giving the play by the re

## 5. Acceptance check

Run this cell — it should pass once `disagreements` is filled in correctly.

In [ ]:
assert len(disagreements) >= 5, "Need at least 5 documented disagreements"
for d in disagreements:
    assert d["text"].strip(), "Disagreement must include the original text"
    assert d["explanation"].strip(), "Disagreement must include an explanation"
    assert d["true_label"] != d["predicted_label"], (
        "A disagreement must have a true_label different from predicted_label"
    )
print(f"OK — {len(disagreements)} disagreements documented.")

OK — 5 disagreements documented.


## RESULTS

```text
**baseline: **
Pretrained model: distilbert-base-uncased-finetuned-sst-2-english
Dataset: IMDb test split
Sample size: 300 reviews, shuffled with seed 42
Device: CPU

**solution:**
Compared the pretrained model's sentiment predictions with the IMDb labels.
Reviews analyzed: 300
Disagreements found: 59
Disagreement rate: 19.67%
The model disagreed with the dataset label on approximately one in five sampled reviews.

**runtime:**
Average inference time per review: 0.2203 seconds
Hardware: CPU

**memory / model size:**
Number of parameters: 66,955,010
Estimated FP32 weight size: 255.41 MiB
Peak RAM usage: Not measured


**3 failure cases:**
ID 2 — IMDb: negative; model: positive.
The review criticizes the film's minimalist style and concludes
that it is nonessential.
The model may have interpreted the descriptive language and
references to the director positively without
capturing the overall negative evaluation.

ID 5 — IMDb: positive; model: negative.
The reviewer praises the film as being
 ahead of its time and calls it
 a “terrific little fantasy.” The model may have
 been influenced by the description of the old,
 grainy television image or other words that do
 not express negative sentiment about the film itself.

ID 11 — IMDb: positive; model: negative.
The reviewer describes the show as great,
praises the actors and comedy, and gives
it a rating of 7/10. The model may have
 overemphasized the criticism of a recurring
 cigarette joke instead of the overall positive opinion.


**what I would try next:**
Investigate mixed sentiment, sarcasm, and potentially ambiguous IMDb labels.
```